# Iris: feature engineering for logistic classification

The Iris dataset contains 150 flowers, 50 from each of three species:
**setosa**, **versicolor**, and **virginica**. Each row records four measurements
in centimeters: sepal length, sepal width, petal length, and petal width. The
sepal is the outer leaf-like part of a flower; the petal is inside it. The
`target` column codes the species as 0, 1, or 2. The dataset is small and
balanced, so its test-set results are useful for learning but imprecise as
evidence about future flowers.

Our question is how the choice of **features** changes a logistic classifier.
First we distinguish versicolor from virginica using two petal measurements;
their measurements overlap, making this more instructive than separating
setosa. We compare a straight decision boundary with one made possible by
quadratic features. Then we classify all three species and compare sepal
measurements with petal measurements. We keep test rows out of preprocessing
and fitting throughout. `load_iris` ships with scikit-learn, so no download
is needed.

### What the three labels mean

These are three iris species, not three quality levels. The photos below
illustrate the species; **the 150 dataset rows contain measurements, not
images**. The model never sees flower color or a photograph.

| 0: setosa | 1: versicolor | 2: virginica |
|:---:|:---:|:---:|
| <img src="https://commons.wikimedia.org/wiki/Special:FilePath/Wild_iris_flower_iris_setosa.jpg?width=500" height="170" alt="Iris setosa flower"> | <img src="https://commons.wikimedia.org/wiki/Special:FilePath/Iris_versicolor.jpg?width=500" height="170" alt="Iris versicolor flower"> | <img src="https://commons.wikimedia.org/wiki/Special:FilePath/Iris_virginica_-_NRCS.jpg?width=500" height="170" alt="Iris virginica flower"> |

Photo sources: [setosa](https://commons.wikimedia.org/wiki/File:Wild_iris_flower_iris_setosa.jpg), U.S. Fish and Wildlife Service (public domain); [versicolor](https://commons.wikimedia.org/wiki/File:Iris_versicolor.jpg), Nichole Ouellette ([CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)); [virginica](https://commons.wikimedia.org/wiki/File:Iris_virginica_-_NRCS.jpg), USDA-NRCS (public domain).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

SEED = 26508
iris = load_iris(as_frame=True)
frame = iris.frame.copy()
print('species:', list(iris.target_names))
print('rows by species:', frame['target'].value_counts().sort_index().to_dict())
X_all = iris.data.to_numpy(float)
y_all = iris.target.to_numpy(int)
setosa_ids = np.flatnonzero(y_all == 0)
binary_ids = np.flatnonzero(y_all != 0)
setosa_train, setosa_test = train_test_split(
    setosa_ids, test_size=.30, random_state=SEED)
binary_train, binary_test = train_test_split(
    binary_ids, test_size=.30, random_state=SEED,
    stratify=y_all[binary_ids])
train_all = np.concatenate([setosa_train, binary_train])
test_all = np.concatenate([setosa_test, binary_test])
print(f'fixed split: {len(train_all)} training flowers, {len(test_all)} test flowers')
frame.head()

### What the measurements show

The four inputs form two pairs: sepal length and width, and petal length and
width. Each point below is **one training flower**. Its position records two
measurements in centimeters; its color gives the known species label. Setosa
petals cluster apart, while versicolor and virginica petals overlap. Sepal
measurements overlap more across the three species.

We split each species into 35 training and 15 test flowers before plotting.
The binary and three-class comparisons reuse those same rows, so all 45 test
flowers remain out of these exploratory plots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, columns, title in [
    (axes[0], (0, 1), 'Sepal measurements'),
    (axes[1], (2, 3), 'Petal measurements'),
]:
    for label, color in [(0, '#507A62'), (1, '#286DC0'), (2, '#B35C62')]:
        take = train_all[y_all[train_all] == label]
        ax.scatter(X_all[take, columns[0]], X_all[take, columns[1]],
                   s=28, alpha=.8, color=color, label=iris.target_names[label])
    ax.set(title=title, xlabel=iris.feature_names[columns[0]],
           ylabel=iris.feature_names[columns[1]])
axes[1].legend(title='species')
fig.tight_layout()
plt.show()

## 1. A fixed binary-classification test set

Keep species 1 (versicolor) and 2 (virginica). In this binary problem,
$Y=0$ means versicolor and $Y=1$ means virginica. A 70/30 stratified split
puts 35 flowers of each species in training and 15 of each in the test set.
We will fit on the 70 training flowers and use the 30 test flowers only after
fitting.

The scatterplot uses **training rows only**. Ask whether a straight line could
separate the two clouds perfectly. The overlapping measurements mean that
classification is uncertain near their meeting point; a fitted probability
can express that uncertainty better than a hard label alone.

In [ ]:
feature_columns = [2, 3]  # petal length and width
features = [iris.feature_names[j] for j in feature_columns]
X = X_all[:, feature_columns]
y = (y_all == 2).astype(int)
train, test = binary_train, binary_test
print(f'train: {len(train)} flowers, {y[train].sum()} virginica')
print(f'test:  {len(test)} flowers, {y[test].sum()} virginica')

training_frame = pd.DataFrame(X[train], columns=features)
training_frame['species'] = np.where(y[train] == 1, 'virginica', 'versicolor')
print('Mean petal measurements in the training set (cm):')
display(training_frame.groupby('species')[features].mean().round(2))

fig, ax = plt.subplots(figsize=(6, 4))
for label, name, color in [(0, 'versicolor', '#286DC0'),
                           (1, 'virginica', '#B35C62')]:
    take = y[train] == label
    ax.scatter(X[train][take, 0], X[train][take, 1],
               label=name, color=color, alpha=.75)
ax.set(xlabel=features[0], ylabel=features[1], title='Training flowers: overlapping petals')
ax.legend(); plt.show()

## 2. Linear and quadratic scores

Write the standardized petal length and width as $z_1,z_2$. The linear model
has score $s(z)=\beta_0+\beta_1z_1+\beta_2z_2$ and probability
$\widehat p(z)=1/(1+e^{-s(z)})$. Its $0.5$ boundary solves $s(z)=0$, a line.
The quadratic model adds $z_1^2$, $z_1z_2$, and $z_2^2$ to the score. It is
still logistic regression in its coefficients, but its boundary can curve in
the original petal-length/petal-width plane.

`make_pipeline` first estimates the scaling from training rows, then
expands the scaled features, then fits logistic regression. Both fits use
the same default $L_2$ regularization strength, `C=1`. The test rows
do not set the scaling, coefficients, or feature choice.

The table reports three different questions: accuracy counts correct labels
at the $0.5$ threshold; ROC AUC asks whether a virginica flower generally
receives a higher score than a versicolor flower; log loss judges the
probabilities themselves, penalizing confident errors strongly. We compare
both models on the *same* 30 held-out flowers.

In [ ]:
models = {
    'linear': make_pipeline(
        StandardScaler(), LogisticRegression(C=1, max_iter=2000)),
    'quadratic': make_pipeline(
        StandardScaler(), PolynomialFeatures(degree=2, include_bias=False),
        LogisticRegression(C=1, max_iter=2000)),
}
rows = []
for name, model in models.items():
    model.fit(X[train], y[train])
    p = model.predict_proba(X[test])[:, 1]
    rows.append({'model': name,
                 'test accuracy': accuracy_score(y[test], p >= .5),
                 'test AUC': roc_auc_score(y[test], p),
                 'test log loss': log_loss(y[test], p)})
results = pd.DataFrame(rows).set_index('model')
display(results.round(3))

Read the table before looking at the boundaries.
One changed test prediction changes accuracy by $1/30\approx0.033$. AUC can
stay the same even when accuracy changes because AUC uses the ranking across
all thresholds, not only the $0.5$ decision. Log loss can change even when
both ranking and decisions agree, because it uses the probability values.

The next figure evaluates each fitted model on a grid of possible petal
measurements. The solid contour is $\widehat p(\mathrm{virginica}\mid x)=0.5$;
the background color is that probability. Only after fitting do we overlay
the held-out test flowers. Compare the shapes of the boundaries and look for
test flowers near them. The grid is for visualization, not extra training
data.

In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-.4, X[:, 0].max()+.4, 250),
                     np.linspace(X[:, 1].min()-.3, X[:, 1].max()+.3, 250))
grid = np.column_stack([xx.ravel(), yy.ravel()])
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, models.items()):
    probability = model.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, probability, levels=np.linspace(0, 1, 11),
                cmap='RdBu_r', alpha=.35)
    ax.contour(xx, yy, probability, levels=[.5], colors=['#202A35'])
    for label, color in [(0, '#286DC0'), (1, '#B35C62')]:
        take = test[y[test] == label]
        ax.scatter(X[take, 0], X[take, 1], color=color, s=22,
                   edgecolor='white', linewidth=.4)
    ax.set(title=name, xlabel=features[0])
axes[0].set_ylabel(features[1])
plt.tight_layout(); plt.show()

## 3. Multiclass logistic regression

Now restore setosa and compare which measurements carry information about
**all three** species. With three classes, the model computes a score
$s_k(x)$ for each species and converts them to probabilities by
$\widehat p_k(x)=e^{s_k(x)}/\sum_{j=0}^2e^{s_j(x)}$. Each row of
`predict_proba` therefore has three nonnegative entries summing to one.
The predicted class has the largest probability.

We fit three models on the same 105 training flowers: one sees only sepal
length and width, one sees only petal length and width, and one sees all four.
All three use the same 45 held-out flowers. Each confusion matrix has
**actual species in rows** and **predicted species in columns**; its diagonal
counts correct classifications. Compare both accuracy and log loss, since
two models can make the same decisions with different confidence.

In [ ]:
multiclass_rows = []
for name, columns in [('two sepal features', [0, 1]),
                      ('two petal features', [2, 3]),
                      ('all four features', [0, 1, 2, 3])]:
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(C=1, max_iter=2000))
    model.fit(X_all[train_all][:, columns], y_all[train_all])
    probabilities = model.predict_proba(X_all[test_all][:, columns])
    predictions = model.predict(X_all[test_all][:, columns])
    assert np.allclose(probabilities.sum(axis=1), 1)
    multiclass_rows.append({'features': name,
                            'accuracy': accuracy_score(y_all[test_all], predictions),
                            'log loss': log_loss(y_all[test_all], probabilities)})
    print(name)
    print(pd.DataFrame(confusion_matrix(y_all[test_all], predictions),
                       index=iris.target_names, columns=iris.target_names))
display(pd.DataFrame(multiclass_rows).set_index('features').round(3))

## What to take away

The binary experiment changes the feature map while keeping the two measured
petal variables and the fitting procedure fixed. The multiclass experiment
changes which measurements the model can see. Read the tables together with
the scatterplot: petal measurements distinguish these species more clearly
than sepal measurements in this dataset, while the quadratic boundary offers
only a modest change on the 30-flower binary test set.

These are small held-out sets: one binary test flower changes accuracy by
3.3 percentage points, and one multiclass test flower changes it by
2.2 points. In the three-class comparison here, petal measurements and all
four measurements make the same number of correct decisions, yet their log
losses differ because their probabilities differ. Neither table establishes
how accurately a model will classify future flowers.

## Try it

Change `feature_columns` in Section 1. Before running the model, predict
whether the boundary and held-out log loss will change. Keep the split fixed
so the comparison isolates your feature choice.